# EDA 01: Data inventory and quality

What is in the lake, how complete it is, and whether every gap is explained.

- **Data:** the DuckDB views in `data/market.duckdb` (rebuild with `python -m src.ingest views`) and the
  download tracker `data/ingest.sqlite`.
- **Period:** the full collected range (2015 → Nov 2026). This notebook checks coverage and quality only;
  no returns or signals are analysed, so using the test period here is allowed.
- **Helpers:** `src/analysis/inventory.py`, `src/analysis/plots.py`.
- **Figures:** `report/figures/eda/01_*.png`.

In [ ]:
import json
import sqlite3
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src.analysis import inventory as inv
from src.analysis.plots import heatmap, save
from src.config import Config

pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_colwidth", 80)
cfg = Config.load()
con = duckdb.connect(str(cfg.duckdb_path), read_only=True)
con.execute("SET TimeZone = 'UTC'")
trk = sqlite3.connect(cfg.tracker_db)

calendar = pd.read_sql("SELECT trade_date, is_trading, note FROM trading_calendar", trk)
DAYS = inv.trading_days(calendar)
universe = pd.read_csv(cfg.manual_dir / "universe.csv")
u50 = universe[universe.tier == "nifty50"]
print(f"trading calendar {DAYS.min().date()} to {DAYS.max().date()}: {len(DAYS)} trading days; "
      f"universe: {len(u50)} Nifty 50 stocks")

## 1. Inventory

One row per lake table. `null_share` lists only key columns that have nulls.

In [ ]:
tables = {name: inv.load_table(con, spec) for name, spec in inv.TABLES.items()}
lake_dirs = sorted(p.name for p in cfg.lake_dir.iterdir() if p.is_dir())
missing_spec = set(lake_dirs) - set(inv.TABLES)
assert not missing_spec, f"lake tables without an inventory spec: {missing_spec}"

inventory = pd.DataFrame([inv.inventory_row(tables[n], inv.TABLES[n]) for n in lake_dirs]).set_index("table")
inventory

## 2. Coverage

### 2a. Prices: share of trading days with a price, stock × year

Counted from each stock's first price, so a stock listed later is blank before its listing, not a gap.

In [ ]:
prices = tables["prices_daily"]
pcov = inv.price_coverage(prices, DAYS)
fig, ax = plt.subplots(figsize=(9, 12))
heatmap(ax, pcov, "prices_daily: share of NSE trading days with a price", cmap="RdYlGn", vmin=0.9, vmax=1,
        cbar_label="share of trading days")
save(fig, "01_price_coverage")
plt.show()
print("stock-years below 100%:", int((pcov < 1).sum().sum()), "| duplicates (symbol, trade_date):",
      int(prices.duplicated(["symbol", "trade_date"]).sum()), "| zero-volume rows:", int((prices.volume == 0).sum()))

### 2b. Announcements, insider trades, shareholding and results: rows per stock × year

In [ ]:
counts = {
    "announcements (published_at)": inv.stock_year_counts(tables["announcements"], "published_at"),
    "insider_trades (acq_to)": inv.stock_year_counts(tables["insider_trades"], "acq_to"),
    "shareholding (quarter end)": inv.stock_year_counts(tables["shareholding"], "period_end"),
    "financial_results (period_to)": inv.stock_year_counts(tables["financial_results"], "period_to"),
}
fig, axes = plt.subplots(2, 2, figsize=(16, 22))
for ax, (title, m) in zip(axes.flat, counts.items()):
    m = m.reindex(sorted(u50.symbol)).fillna(0)
    heatmap(ax, m.where(m > 0), title + ": rows (blank = none)", cmap="viridis", cbar_label="rows")
fig.tight_layout()
save(fig, "01_event_coverage")
plt.show()

per_year = pd.DataFrame({t.split(" ")[0]: m.sum() for t, m in counts.items()}).fillna(0).astype(int)
per_year["insider_stocks"] = (counts["insider_trades (acq_to)"] > 0).sum()
per_year

### 2c. Index and macro series: share of NSE trading days with a value, per year

Macro series are US market data: US holidays have no value, so ~95% is the expected level.

In [ ]:
idx = tables["index_daily"]; macro = tables["macro"]
start = DAYS[DAYS >= "2016-01-01"].min(); end = prices.trade_date.max()
series = {name: inv.series_day_coverage(g.trade_date, DAYS, start, end) for name, g in idx.groupby("index_name")}
# FRED keeps a row on US holidays with an empty value, so count non-null values only
series |= {sid: inv.series_day_coverage(g.obs_date[g.value.notna()], DAYS, start, end)
           for sid, g in macro.groupby("series_id")}
scov = pd.DataFrame(series).T
fig, ax = plt.subplots(figsize=(10, 3.2))
heatmap(ax, scov, "Share of NSE trading days with a value", cmap="RdYlGn", vmin=0.8, vmax=1, fmt="{:.0%}")
save(fig, "01_series_coverage")
plt.show()

last = pd.concat([idx.groupby("index_name").trade_date.max(), macro.groupby("series_id").obs_date.max()])
display(pd.DataFrame({"last_value": last.astype(str), "days_behind_prices": (pd.Timestamp(end) - pd.to_datetime(last)).dt.days}))

## 3. Download tracker

Tasks per source and status from `data/ingest.sqlite`, then the open review items.

In [ ]:
tasks = pd.read_sql("SELECT task_id, source, symbol, period_start, status FROM tasks", trk)
status = tasks.pivot_table(index="source", columns="status", values="task_id", aggfunc="count", fill_value=0)
status["total"] = status.sum(axis=1)
display(status)

ax = status.drop(columns="total").plot.barh(stacked=True, figsize=(9, 5), colormap="tab10")
ax.set_xscale("log"); ax.set_xlabel("tasks (log scale)"); ax.set_title("Tracker tasks per source and status")
save(ax.figure, "01_tracker_status")
plt.show()

reviews = pd.read_sql("""SELECT r.review_id, t.source, t.symbol, r.check_name, r.severity, r.details, r.status
                         FROM review_items r JOIN tasks t USING (task_id)""", trk)
print("review items by status:", reviews.status.value_counts().to_dict())
display(reviews[reviews.status == "open"].groupby(["source", "check_name", "severity"]).size().rename("open_items").to_frame())
adj = reviews[(reviews.status == "open") & (reviews.check_name == "manual_adjustment_needed")]
display(pd.DataFrame([{"symbol": s, **json.loads(d)} for s, d in zip(adj.symbol, adj.details)]).sort_values("ex_date"))

cov_tbl = pd.read_sql("SELECT source, status, COUNT(*) n, MIN(trade_date) first, MAX(trade_date) last FROM coverage GROUP BY 1, 2", trk)
cov_tbl

## 4. Timestamps

The forecast for day *d* is made at 08:30 IST. A record published before 08:30 IST on a trading day can
inform that day's trade; anything later informs the next trading day. A record with only a date (time
00:00) can't be placed safely and would have to be pushed to the next day.

In [ ]:
ts_cols = [("announcements", "published_at"), ("financial_results", "broadcast_at"),
           ("financial_results", "published_at"), ("board_meetings", "published_at"),
           ("insider_trades", "published_at"), ("corporate_actions", "published_at"),
           ("shareholding", "broadcast_ist"), ("news", "published_at"), ("macro", "known_from")]
rows = []
for t, c in ts_cols:
    s = tables[t][c]
    if not isinstance(s.dtype, pd.DatetimeTZDtype):     # JSON broadcast times are IST wall-clock
        s = pd.to_datetime(s).dt.tz_localize(inv.IST)
    sess = inv.ist_session(s).value_counts(normalize=True)
    rows.append({"table": t, "column": c, "rows": len(s), "null": s.isna().mean(), "has_time": inv.has_time(s).mean(),
                 **{k: sess.get(k, 0.0) for k in inv.SESSIONS}})
tstab = pd.DataFrame(rows).set_index(["table", "column"])
tstab.style.format({c: "{:.1%}" for c in ["null", "has_time", *inv.SESSIONS]})

In [ ]:
ann = tables["announcements"]; res = tables["financial_results"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (name, s) in zip(axes, [("announcements.published_at", ann.published_at),
                                ("financial_results.broadcast_at", res.broadcast_at)]):
    t = s.dropna().dt.tz_convert(inv.IST)
    hrs = t.dt.hour + t.dt.minute / 60
    ax.hist(hrs, bins=96, range=(0, 24), color="#4C72B0")
    for x in (8.5, 15.5):
        ax.axvline(x, color="crimson", ls="--", lw=1)
    ax.set_title(f"{name}: time of day (IST), n={len(t):,}", fontsize=10)
    ax.set_xlabel("hour of day, IST (dashed: 08:30 forecast, 15:30 close)")
    ax.set_xticks(range(0, 25, 2))
fig.tight_layout()
save(fig, "01_timestamps")
plt.show()

def informs(s):
    s = s.dropna()
    day = inv.first_informed_day(s, DAYS)
    same = day.dt.date.values == s.dt.tz_convert(inv.IST).dt.date.values
    return pd.Series({"records": len(s), "same_day": same.mean(), "next_trading_day_or_later": 1 - same.mean(),
                      "beyond_calendar": day.isna().mean()})

pd.DataFrame({"announcements": informs(ann.published_at), "results": informs(res.broadcast_at)}).T

In [ ]:
# Share of announcements with a real time, per year (a check for date-only years)
yr = ann.published_at.dt.tz_convert(inv.IST).dt.year
pd.DataFrame({"rows": yr.value_counts().sort_index(),
              "has_time": inv.has_time(ann.published_at).groupby(yr).mean().round(4)}).T

## 5. Gap reconciliation

Each check below feeds one line of the reconciliation table at the end.

In [ ]:
# 5a. Calendar vs prices
missing_days, extra_days = inv.calendar_mismatch(prices.trade_date, DAYS)
weekday_extra = [d for d in extra_days if d.weekday() < 5]
print("trading days with no prices:", len(missing_days), "| price days the calendar marks closed:", extra_days)

# 5b. Stocks that start after 1 Jan 2016
late = inv.late_starts(prices, universe, "2016-01-04")
display(late)

# 5c. Raw moves above 20%
moves = inv.large_moves(prices, tables["corporate_actions"])
print("moves > 20%:", len(moves), "| on a split/bonus/rights/demerger ex-date:", int(moves.explained.sum()))
display(moves[~moves.explained][["symbol", "trade_date", "prev_close", "close", "ret"]])

# 5d. prev_close in the file vs the previous row's close
p = prices.sort_values(["symbol", "trade_date"]).copy()
p["lag_close"] = p.groupby("symbol").close.shift()
pc = p[(p.prev_close / p.lag_close - 1).abs() > 0.01]
ca_days = set(zip(tables["corporate_actions"].symbol, pd.to_datetime(tables["corporate_actions"].ex_date)))
pc_on_ca = sum((s, pd.Timestamp(d)) in ca_days for s, d in zip(pc.symbol, pc.trade_date))
print(f"prev_close differs from previous close by >1%: {len(pc)} rows, {pc_on_ca} on a corporate-action ex-date")
display(pc.loc[[(s, pd.Timestamp(d)) not in ca_days for s, d in zip(pc.symbol, pc.trade_date)],
               ["symbol", "trade_date", "lag_close", "prev_close", "close"]])

In [ ]:
# 5e. Tracker no_data tasks: holidays, or stock-years before listing?
nd = tasks[tasks.status == "no_data"].copy()
nd["period_start"] = pd.to_datetime(nd.period_start)
daily = nd[nd.source.isin(["nse_bhavcopy", "nse_indices"])]
print("bhavcopy/index no_data on a calendar trading day:", int(daily.period_start.isin(DAYS).sum()), "of", len(daily))

first_price = pd.to_datetime(prices.groupby("symbol").trade_date.min())
yearly = nd[nd.source.isin(["nse_announcements", "nse_insider"])].copy()
yearly["year"] = yearly.period_start.dt.year
yearly["before_listing"] = [y < first_price.get(s, pd.Timestamp("2016-01-01")).year for s, y in zip(yearly.symbol, yearly.year)]
display(yearly.groupby(["source", "before_listing"]).size().rename("no_data_tasks").to_frame())
display(yearly[(yearly.source == "nse_announcements") & ~yearly.before_listing][["symbol", "year"]])
print("insider no_data stock-years by year:", yearly[yearly.source == "nse_insider"].year.value_counts().sort_index().to_dict())
print("other no_data:", nd[~nd.source.isin(["nse_bhavcopy", "nse_indices", "nse_announcements", "nse_insider"])]
      .groupby("source").symbol.apply(list).to_dict())

In [ ]:
# 5f. Small or partial tables
fin = tables["financial_results"]
print("results: last quarter", fin.period_to.max(), "| rows without broadcast_at:", int(fin.broadcast_at.isna().sum()),
      f"({fin.broadcast_at.isna().mean():.1%}); their period_to range:",
      fin.loc[fin.broadcast_at.isna(), "period_to"].min(), "to", fin.loc[fin.broadcast_at.isna(), "period_to"].max())
txt = tables["announcement_text"]
print("announcement_text:", len(txt), "docs,", txt.symbol.nunique(), "stocks, published",
      txt.published_at.min(), "to", txt.published_at.max(), "| scanned:", int(txt.scanned.sum()))
print("news:", len(tables["news"]), "rows,", tables["news"].symbol.nunique(), "stocks, months", tables["news"].month.unique())
print("fii_dii:", len(tables["fii_dii"]), "rows on", tables["fii_dii"].trade_date.unique())
print("pledges:", len(tables["pledges"]), "rows, stocks", sorted(tables["pledges"].symbol))
print("rbi_policy.csv rows:", len(pd.read_csv(cfg.manual_dir / "rbi_policy.csv")))
print("corporate_actions published_at null:", f"{tables['corporate_actions'].published_at.isna().mean():.0%}")
bm = tables["board_meetings"]
print("board_meetings per stock: min", bm.groupby("symbol").size().min(), "max", bm.groupby("symbol").size().max(),
      "| earliest meeting", bm.meeting_date.min())
sh = tables["shareholding"]
print("shareholding quarters per stock: min", sh.groupby("symbol").period_end.nunique().min(),
      "max", sh.groupby("symbol").period_end.nunique().max(), "| range", sh.period_end.min(), "to", sh.period_end.max())
sh_stocks = sh.groupby(sh.period_end.dt.year).symbol.nunique()
print("shareholding: stocks with a quarter, per year:", sh_stocks.to_dict())
no_insider = sorted(set(u50.symbol) - set(tables["insider_trades"].symbol))
no_results = sorted(set(u50.symbol) - set(fin.symbol))
print("stocks with no insider rows at all:", no_insider, "| with no results:", no_results)

In [ ]:
# 5h. A trading day missing from both the calendar and the data?
# NSE's own prev_close (prices) and points_change (index) say what the previous session closed at.
# If they disagree with our previous row on the same day for every stock, a session is missing in between.
by_day = pc.groupby("trade_date").symbol.nunique()
full_day_breaks = by_day[by_day >= 0.8 * prices.symbol.nunique()]
nifty = idx[idx.index_name == "Nifty 50"].sort_values("trade_date").copy()
nifty["implied_prev"] = nifty.close - nifty.points_change
nifty["lag_close"] = nifty.close.shift()
nbreak = nifty[(nifty.implied_prev / nifty.lag_close - 1).abs() > 0.005]
print("days where >=80% of stocks' prev_close disagrees with our previous close:", [d.date() for d in full_day_breaks.index])
display(nbreak[["trade_date", "lag_close", "implied_prev", "close", "points_change"]])
print(calendar[(calendar.trade_date >= "2026-01-30") & (calendar.trade_date <= "2026-02-02")])

In [ ]:
# 5g. Reconciliation table: every gap or oddity, matched to a known item or NEW
unexplained = moves[~moves.explained]
listing_day = unexplained[[ (s, d) in {(r.symbol, pd.Timestamp(r.first_price)) for r in late.itertuples()}
                           for s, d in zip(unexplained.symbol, unexplained.trade_date)]]
n_txt = len(tables["announcement_text"])
budget = nbreak[pd.to_datetime(nbreak.trade_date) == pd.Timestamp("2026-02-02")]
gaps = pd.DataFrame([
    ("Bhavcopy/index: no_data tasks", f"{len(daily)} tasks, {int(daily.period_start.isin(DAYS).sum())} on trading days",
     "Expected: holidays/weekends (tracker rule 4)", "known"),
    ("Index file 20 Jun 2016 missing", "filled from yfinance", "Backfill result note", "known"),
    ("JIOFIN first price 4 Sep 2023, not 21 Aug", f"first price {first_price['JIOFIN'].date()}", "Backfill result note (BE series)", "known"),
    ("Stocks listed after 2016 (BSE, SBILIFE, HDFCLIFE, MAXHEALTH, ETERNAL)", "first price 2017–2021",
     "Genuine listings (ETERNAL = ZOMATO, open item 4)", "known"),
    ("TATACONSUM prices start 27 Feb 2020", "no history as TATAGLOBAL (renamed Feb 2020)", "Open item 4 asks to look for others", "NEW"),
    ("SHRIRAMFIN prices start 20 Dec 2022", "no history as SRTRANSFIN (renamed Dec 2022)", "Open item 4 asks to look for others", "NEW"),
    ("Sunday 1 Feb 2026 Budget session missing from calendar, tracker and data",
     f"prev_close disagrees for {int(by_day.max())} of 50 stocks on 2 Feb 2026; Nifty implied previous close "
     f"{budget.implied_prev.iloc[0]:,.2f} vs our {budget.lag_close.iloc[0]:,.2f}" if len(budget) else "not found",
     "calendar_overrides.csv has no 2026-02-01 row", "NEW"),
    ("Nifty change columns off on 13 Mar 2023", "points_change -435.30 is measured from the 9 Mar close; "
     "closes are right (10 Mar 17,412.90, 13 Mar 17,154.30)", "NSE file quirk; use close-to-close, not change_pct", "NEW"),
    ("Weekday Muhurat sessions marked closed in trading_calendar", f"{len(weekday_extra)} days with prices: "
     + ", ".join(map(str, weekday_extra)), "calendar_overrides.csv lists only weekend sessions", "NEW"),
    ("Raw moves >20% on corporate-action days", f"{int(moves.explained.sum())} of {len(moves)}", "Splits/bonuses (task 2.4); open item 13", "known"),
    ("Raw moves >20% on listing days", f"{len(listing_day)}: " + ", ".join(listing_day.symbol), "prev_close is the issue price", "known"),
    ("Raw moves >20% with no corporate action", f"{len(unexplained) - len(listing_day)} rows (Mar 2020 crash, Adani Feb 2023 / Nov 2024, "
     "4 Jun 2024 election, SBIN Oct 2017 recap…)", "Market moves; ADANIENT 2017-04-26, 2019-05-20, 2020-08-25 not yet attributed", "NEW"),
    ("15 rights issues / demergers need a manual factor", f"{len(adj)} open review items", "Open item 13 (task 2.4)", "known"),
    ("Announcements: no_data stock-years before listing", f"{int(((yearly.source == 'nse_announcements') & yearly.before_listing).sum())} tasks",
     "Genuine: stock not listed yet", "known"),
    ("NESTLEIND: no announcements 2016–2022", f"{int(((yearly.source == 'nse_announcements') & ~yearly.before_listing).sum())} "
     "no_data stock-years for a stock listed throughout", "Not in open items", "NEW"),
    ("Stocks with no insider rows at all", ", ".join(no_insider), "Open item 12 covers thinning, not whole stocks", "NEW"),
    ("Stocks with no results rows", ", ".join(no_results) + " (insurers)", "Not in open items", "NEW"),
    ("Shareholding: almost nothing before 2021", f"stocks with a quarter: {int(sh_stocks.get(2020, 0))} in 2020, "
     f"{int(sh_stocks.get(2021, 0))} in 2021", "Step 8 says 'check how much history' only for pledges", "NEW"),
    ("Insider trades thin out after 2021", f"{int((yearly.source == 'nse_insider').sum())} no_data stock-years; "
     f"{int(per_year.loc[2016, 'insider_trades'])} rows in 2016 vs {int(per_year.loc[2025, 'insider_trades'])} in 2025", "Open item 12", "known"),
    ("TCS promoter sale Mar 2024 missing from insider data", "", "Open item 9 (SAST)", "known"),
    ("Pledges: every task no_data", f"{len(tables['pledges'])} rows (pilot only)", "Open item 11", "known"),
    ("Results list stops at Dec 2024 quarter, no revenue/profit", f"last period_to {fin.period_to.max()}", "Open item 1; review results_list_stale", "known"),
    ("Results rows without broadcast time", f"{int(fin.broadcast_at.isna().sum())} rows ({fin.broadcast_at.isna().mean():.1%}), "
     f"periods {fin.loc[fin.broadcast_at.isna(), 'period_to'].min()} to {fin.loc[fin.broadcast_at.isna(), 'period_to'].max()}",
     "Not in open items; no impact (all before 2016)", "NEW"),
    ("Corporate actions have no published_at", f"{tables['corporate_actions'].published_at.isna().mean():.0%} null",
     "Not in open items (ex-dates are announced in advance; announcement time unknown)", "NEW"),
    ("Board meetings: only the latest ~20 per stock", f"{len(bm)} rows, earliest {bm.meeting_date.min()}", "Open item 6", "known"),
    ("Shareholding: limited history", f"{sh.period_end.min()} to {sh.period_end.max()}", "Step 8 (endpoint returns recent quarters)", "known"),
    ("FII/DII: 2 rows", "daily job only", "Open item 2", "known"),
    ("RBI policy file empty", "0 rows", "Review empty_manual_file; task 4.3", "known"),
    ("News: pilot only", f"{len(tables['news'])} rows, June 2024", "Open items 5 and 8", "known"),
    ("Announcement PDFs: pilot only", f"{n_txt} docs, 5 stocks, 2024", "Step 7 full run not done; open item 10", "known"),
    ("Macro series end before prices", "VIXCLS/DGS10 5 Nov, DEXINUS/DTWEXBGS 30 Oct", "FRED publication lag (step 10)", "known"),
    ("Tracker coverage table built for June 2024 only", f"{int(cov_tbl.n.sum())} rows", "Not rebuilt after the backfill: run `python -m src.ingest coverage`", "NEW"),
    ("DuckDB views pointed to another machine's path", "13 views failed until rebuilt", "Views store absolute paths; rebuild per machine", "NEW"),
], columns=["gap_or_oddity", "evidence", "matched_to", "status"])
print("NEW:", int((gaps.status == "NEW").sum()), "of", len(gaps))
gaps.style.apply(lambda r: ["background-color: #fde2e1" if r.status == "NEW" else "" for _ in r], axis=1)

## Findings

1. **13 lake tables, all inventoried.** Prices hold 125,999 rows (50 stocks, 1 Jan 2016 – 6 Nov 2026); announcements 74,876 (50 stocks); insider trades 100,848 (46 stocks); results 5,625 (48 stocks); shareholding 1,081; macro 14,140 (5 series). Four tables are pilot-sized: news 14, announcement_text 14, pledges 5, fii_dii 2.
2. **Prices are complete against the calendar.** 0 of the 2,681 calendar trading days lack prices; every stock-year is at 100% from the stock's first day. There are 0 duplicate (symbol, date) rows and 0 zero-volume rows.
3. **One whole session is missing from calendar, tracker and data: Sunday 1 Feb 2026 (Union Budget).** On 2 Feb 2026, NSE's prev_close disagrees with our previous close for 41 of 50 stocks. The Nifty's implied previous close is 24,825.45, against our 25,320.65 for 30 Jan. This is in the test period.
4. **The trading calendar marks 6 weekday Muhurat sessions as closed:** 19 Oct 2017, 7 Nov 2018, 4 Nov 2021, 24 Oct 2022, 1 Nov 2024 and 21 Oct 2025. Prices exist for those days. Anything that counts trading days (horizons, event days, coverage) is off by one around Diwali.
5. **Two stocks lack their pre-rename history.** TATACONSUM starts on 27 Feb 2020 (it was TATAGLOBAL before) and SHRIRAMFIN on 20 Dec 2022 (SRTRANSFIN). That is about 4 and 7 years of training data lost. The five other late starts (BSE, SBILIFE, HDFCLIFE, MAXHEALTH, ETERNAL) are genuine listings.
6. **49 raw moves above 20%.** 33 fall on split, bonus or demerger ex-dates and 2 are listing days (BSE, ETERNAL). The other 14 are market moves: the 23 Mar 2020 crash, Adani in Feb 2023 and Nov 2024, the 4 Jun 2024 election result, SBIN's Oct 2017 recapitalisation. Three ADANIENT moves (26 Apr 2017, 20 May 2019, 25 Aug 2020) are not yet attributed. The 15 rights issues and demergers still need manual factors (open item 13).
7. **Event data is thinner than prices:**
   - NESTLEIND has 0 announcements for 2016–2022.
   - BSE, ETERNAL, NESTLEIND and POWERGRID have no insider rows at all. Insider rows fall from 19,908 (2016) to 1,467 (2025).
   - HDFCLIFE and SBILIFE have no results rows.
   - Shareholding covers 48–50 stocks only from 2021, with 1 stock or fewer per year before that.
   - The results list ends at the Dec 2024 quarter.
   - Board meetings go back only to 26 Apr 2024.
8. **Timestamps are usable for the time rule.** 99.99% of announcements and 100% of results broadcast times carry a real time. Results rows without a time are all from 2005–2007. In IST, 1.5% of announcements and 1.0% of results arrive before 08:30, 34.6% / 39.9% during market hours and 63.9% / 59.1% after 15:30. So 98.8% of announcements and 99.6% of results first inform the next trading day.
9. **Corporate actions have no published_at (100% null).** Ex-dates are known in advance, but when the action was announced is unknown. A catalyst calendar can't place them in time from this table alone.
10. **Index and macro series:**
    - Nifty 50 and India VIX cover every trading day.
    - The FRED series have a value on 95–98% of NSE trading days, 2016–2026 (US holidays are kept as rows with no value, 4% of macro rows). They end 1–7 days before the last price.
    - The Nifty file for 13 Mar 2023 measures points_change from the 9 Mar close, not the 10 Mar close. Use closes, not change_pct.
11. **The tracker is mostly clean.** 0 bhavcopy or index no_data tasks fall on a trading day (306 tasks, all holidays). There are 64 open review items (15 manual adjustments, 48 stale results lists, 1 empty RBI file) and none blocked. The tracker's coverage table was built only for June 2024 (1,045 rows) and needs `python -m src.ingest coverage`.
12. **Reconciliation: 33 gaps or oddities listed, 14 marked NEW** (see 5g). The NEW items are added to "Open items" in docs/DATA_COLLECTION.md.